In [1]:
file_name = "stations.json"
target_schema = "air_quality"
target_table = "stations"
load_mode = "overwrite"



StatementMeta(, 9a67e96b-7595-4263-a6ce-7cf863e609c5, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql import functions as F

# 1. Читаємо JSON-масив у багаторядковому режимі відносним шляхом з lh_bronze
raw_path = f"Files/raw/{file_name}"
df_raw = spark.read.option("multiline", "true").json(raw_path)

# 2. Додаємо колонку _ingested_at із часом завантаження
df_to_write = df_raw.withColumn("_ingested_at", F.current_timestamp())

# 3. Створюємо цільову схему, якщо її ще немає
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_schema}")

# 4. Записуємо дані у Delta-таблицю у вказаному режимі (overwrite або append)
full_table_name = f"{target_schema}.{target_table}"
df_to_write.write.format("delta").mode(load_mode).saveAsTable(full_table_name)

# 5. Рахуємо кількість записаних цим запуском рядків і оновлюємо таблицю керування
# (Цей блок виконається лише якщо запис вище завершився без помилок)
rows_written = df_to_write.count()

safe_file_name = file_name.replace("'", "''")
spark.sql(f"""
    UPDATE control.files_to_load
    SET last_loaded_at = current_timestamp(),
        last_rows = {rows_written}
    WHERE file_name = '{safe_file_name}'
""")

# 6. Виводимо перевірку результатів
display(spark.table(full_table_name).limit(5))
display(spark.table("control.files_to_load"))


StatementMeta(, 9a67e96b-7595-4263-a6ce-7cf863e609c5, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 053d6e46-d4a1-4ac0-ad88-32d00e10345c)

SynapseWidget(Synapse.DataFrame, 75d9aa4d-3e32-4f74-8db4-2f2ab882b947)